# Chapter 9 - Lab 1: <font color='blue'>Sequential Warranty-Claims Pipeline with a Compliance Guardrail</font> (customer-care edition)

**<font color='purple'>Goal</font>**:
Build a **7-agent sequential pipeline** that processes a NovaMart **Care+ extended-warranty
claim** from intake to final compliance review. This is the **Sequential Pipeline with
Guardrails** architecture from Chapter 7 applied to after-sales service.

The customer: **Ananya Iyer** (C1003, Chennai). Her AquaPure RO water purifier (bought
November 2024, Rs 18,500) stopped dispensing water and is leaking. Her 1-year maker warranty
has ended, but she bought a Care+ plan that runs until November 2027.

Seven agents pass control via explicit handoffs, share data through a typed `Context`
object, and write to an **audit trail** at every step. The seventh agent, Compliance, has
*veto power* and can block a decision that fails the consumer-protection checks.

| Finance (insurance) edition | Customer-care edition |
|---|---|
| FNOL agent (first notice of loss) | **IntakeAgent** (register the warranty claim) |
| Document parser (claim form, photos, police report) | **EvidenceParser** (claim form, photos, technician job sheet) |
| Coverage agent (policy period, exclusions) | **CoverageAgent** (plan period, exclusions such as physical damage) |
| Fraud screener | **AbuseScreener** (cover bought for a known fault, many prior claims, late reporting) |
| Damage assessment (repair estimate - deductible) | **AssessmentAgent** (repair estimate - service fee, capped at the plan limit) |
| Decision agent | **DecisionAgent** (APPROVED / DENIED / ESCALATED) |
| Regulatory compliance agent | **ComplianceAgent** (a denial must cite evidence-backed reasons, etc.) |

**<font color='purple'>Tech stack</font>**:

* **LlamaIndex**: `FunctionAgent`, `AgentWorkflow`, `Context`.
* **OpenAI** `gpt-4o`.
* **Pydantic**: typed contracts for the claim record and helpers from `common.py`.

## 1. Install packages

In [ ]:
%pip install -q llama-index llama-index-llms-openai pydantic python-dotenv

## 2. Set up the OpenAI API key

In [ ]:
import os

try:
    from google.colab import userdata  # Colab: keys live under the key icon
    os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY') or ''
except ImportError:
    # Running locally: read keys from the environment or a .env file.
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass

assert os.environ.get('OPENAI_API_KEY'), 'Set OPENAI_API_KEY first (Colab secret or .env)'

## 3. Bootstrap the shared models and helpers

Chapter 9 labs share a `common.py` module with the Pydantic models, helpers and sample data. If you cloned the repository it is already on disk; in Colab upload it next to the notebook (or set `CC_RAW_BASE`).

In [ ]:
import os, urllib.request

# If you cloned the repository, the helper files already sit next to this notebook.
# In Colab, either upload them with the Files panel or set RAW_BASE to the raw URL
# of the "customer-care-ai-agents" folder in your copy of the repository.
RAW_BASE = os.environ.get("CC_RAW_BASE", "")   # e.g. https://raw.githubusercontent.com/<you>/<repo>/main/customer-care-ai-agents
for name in ['common.py']:
    if not os.path.exists(name):
        if not RAW_BASE:
            raise FileNotFoundError(f"{name} not found: upload it next to the notebook or set CC_RAW_BASE")
        urllib.request.urlretrieve(f"{RAW_BASE}/Chapter%209/{name}", name)
print("Helper files ready:", ['common.py'])

In [ ]:
import json

from common import (
    ProductCategory, ClaimStatus,
    SAMPLE_CLAIM, SAMPLE_PLAN, SAMPLE_DOCUMENTS,
    generate_claim_id, log_audit, days_between,
)

print('Common helpers loaded.')
print(json.dumps(SAMPLE_CLAIM, indent=2))

## 4. Define the seven pipeline tools

Each agent has a dedicated async tool that reads from and writes to the shared `Context`.
Every write also calls `log_audit` to record what happened, when, and which agent did it.

**Business point:** the audit log is the most important artifact here. If Ananya disputes the
outcome, or a consumer forum asks NovaMart to justify a refusal, every step must be
reconstructable from this log.

In [ ]:
from llama_index.core.workflow import Context


def classify_product(description: str) -> ProductCategory:
    """Classify the product category from the description (simplified)."""
    d = description.lower()
    if any(w in d for w in ["refrigerator", "fridge", "air conditioner", " ac ",
                            "washing machine", "washer"]):
        return ProductCategory.LARGE_APPLIANCE
    if any(w in d for w in ["tv", "television", "phone", "laptop", "screen"]):
        return ProductCategory.ELECTRONICS
    if any(w in d for w in ["purifier", "mixer", "grinder", "microwave", "oven"]):
        return ProductCategory.KITCHEN
    return ProductCategory.ACCESSORY


async def process_intake(ctx: Context, claim_data: dict) -> str:
    """Create the warranty-claim record from the customer's request.

    Used by: IntakeAgent (root agent).
    Writes: claim record to shared state.
    """
    if isinstance(claim_data, str):  # some models pass the JSON as a string
        claim_data = json.loads(claim_data)
    claim_id = generate_claim_id()
    category = classify_product(claim_data.get("description", ""))

    claim = {
        "claim_id": claim_id,
        "customer_name": claim_data.get("name", ""),
        "plan_number": claim_data.get("plan_number", ""),
        "product_category": category.value,
        "date_of_failure": claim_data.get("date_of_failure", ""),
        "description": claim_data.get("description", ""),
        "status": "received",
        "parsed_documents": [],
        "audit_log": [],
    }

    state = await ctx.store.get("state")
    state["claim"] = claim
    log_audit(state, "IntakeAgent", "Claim created",
              f"ID: {claim_id}, Category: {category.value}")
    await ctx.store.set("state", state)
    return f"Intake processed. Claim {claim_id}. Category: {category.value}."


async def parse_evidence(ctx: Context) -> str:
    """Parse submitted evidence (claim form, photos, technician job sheet).

    Used by: EvidenceParser.
    Reads: submitted_documents from state.
    Writes: parsed_documents, fault_description to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    documents = state.get("submitted_documents", [])

    parsed, fault_parts = [], []
    for doc in documents:
        doc_type = doc.get("type", "unknown")
        if doc_type == "photo":
            # In production: call a vision model on the photo
            desc = doc.get("description", "Fault visible in photo.")
            parsed.append({"type": "fault_photo", "fault_description": desc})
            fault_parts.append(desc)
        elif doc_type == "claim_form":
            parsed.append({"type": "claim_form", "fields": doc.get("fields", {})})
        elif doc_type == "technician_report":
            summary = doc.get("summary", "")
            parsed.append({"type": "technician_report", "summary": summary})
            fault_parts.append(summary)

    claim["parsed_documents"] = parsed
    claim["fault_description"] = " ".join(fault_parts)
    claim["status"] = "parsing"
    log_audit(state, "EvidenceParser",
              f"Parsed {len(parsed)} documents",
              f"Types: {[d['type'] for d in parsed]}")
    await ctx.store.set("state", state)
    return f"Parsed {len(parsed)} documents."

**Coverage Agent.** Checks that the failure falls inside the plan period and is not excluded (physical damage, unauthorised repair, commercial use, tampered seal). It writes back the coverage flag, limit, service fee, reason codes and reasoning.

In [ ]:
def coverage_evidence(claim: dict, plan: dict) -> tuple[list[str], bool]:
    """Return supported denial codes and whether basic lab evidence is complete.

    Exclusion matching is a simplified lab rule, not a legal coverage assessment.
    """
    from datetime import date

    try:
        failure = date.fromisoformat(claim["date_of_failure"])
        start = date.fromisoformat(plan["inception_date"])
        end = date.fromisoformat(plan["expiration_date"])
    except (KeyError, TypeError, ValueError):
        return [], False
    if start > end:
        return [], False
    codes = []
    if not start <= failure <= end:
        codes.append("OUTSIDE_PLAN_PERIOD")
    exclusions = plan.get("exclusions")
    if not isinstance(exclusions, list) or not all(isinstance(x, str) and x.strip() for x in exclusions):
        return codes, False
    description = claim.get("description", "")
    if not isinstance(description, str) or not description.strip():
        return codes, False
    if any(x.lower() in description.lower() for x in exclusions):
        codes.append("PLAN_EXCLUSION")
    return codes, True


async def verify_coverage(ctx: Context) -> str:
    """Check whether the warranty plan covers this failure.

    Used by: CoverageAgent.
    Reads: claim, plan from state.
    Writes: coverage fields to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    plan = state.get("plan", {})

    reason_codes, evidence_complete = coverage_evidence(claim, plan)
    is_covered = evidence_complete and not reason_codes
    exclusions, reasoning_parts = [], []
    if not evidence_complete:
        reasoning_parts.append("Coverage evidence is incomplete or invalid; human review required.")

    if "OUTSIDE_PLAN_PERIOD" in reason_codes:
        reasoning_parts.append(
            f"Failure date {claim.get('date_of_failure', '')} outside plan period "
            f"({plan.get('inception_date', '')} to {plan.get('expiration_date', '')})."
        )

    plan_exclusions = plan.get("exclusions", [])
    description = claim.get("description", "")
    for exclusion in (plan_exclusions if isinstance(plan_exclusions, list) else []):
        if not isinstance(exclusion, str) or not exclusion.strip():
            continue
        if isinstance(description, str) and exclusion.lower() in description.lower():
            exclusions.append(exclusion)
    if exclusions:
        reasoning_parts.append(f"Exclusions found: {exclusions}")

    if is_covered and not exclusions:
        reasoning_parts.append(
            f"Failure covered under {plan.get('coverage_sections', 'the warranty plan')}. "
            f"Limit: Rs {plan.get('coverage_limit', 0):,.0f}. "
            f"Service fee: Rs {plan.get('service_fee', 0):,.0f}."
        )

    claim["coverage_verified"] = is_covered and not exclusions
    claim["applicable_limit"] = plan.get("coverage_limit", 0)
    claim["service_fee"] = plan.get("service_fee", 0)
    claim["exclusions_found"] = exclusions
    claim["coverage_reasoning"] = " ".join(reasoning_parts)
    claim["coverage_reason_codes"] = reason_codes
    claim["coverage_evidence_complete"] = evidence_complete
    claim["status"] = "coverage_check"

    log_audit(state, "CoverageAgent",
              f"Coverage: {'verified' if claim['coverage_verified'] else 'denied'}",
              claim["coverage_reasoning"])
    await ctx.store.set("state", state)
    return f"Coverage {'verified' if claim['coverage_verified'] else 'not verified'}."

**Abuse Screener.** Adds points for three classic red flags:

* the failure happened within 30 days of the plan starting (was the cover bought for a known fault?),
* three or more prior claims in 12 months,
* the claim was reported more than 30 days after the failure.

A score above 40 sends the claim to the deeper abuse investigation (Lab 2).

In [ ]:
async def screen_for_abuse(ctx: Context) -> str:
    """Screen the claim against warranty-abuse indicators.

    Used by: AbuseScreener.
    Reads: claim, plan, claims_history from state.
    Writes: abuse_score, red_flags, abuse_recommendation to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    plan = state.get("plan", {})
    red_flags, score = [], 0

    # Red flag 1: failure soon after the plan started (cover bought for a known fault?)
    failure = claim.get("date_of_failure", "")
    inception = plan.get("inception_date", "")
    if failure and inception:
        days = days_between(inception, failure)
        if days < 0:
            red_flags.append(f"Failure predates plan start by {-days} days")
            score += 25
        elif days < 30:
            red_flags.append(f"Failure {days} days after plan start")
            score += 25

    # Red flag 2: many prior warranty or 'damaged' claims
    prior_claims = state.get("claims_history", [])
    if len(prior_claims) >= 3:
        red_flags.append(f"{len(prior_claims)} prior claims in 12 months")
        score += 20

    # Red flag 3: late reporting (the cost estimate does not exist yet, so
    # amount-versus-limit checks belong after assessment, not here)
    if failure:
        delay = days_between(failure, state.get("intake_date", failure))
        if delay < 0:
            red_flags.append(f"Claim reported {-delay} days before the recorded failure date")
            score += 15
        elif delay > 30:
            red_flags.append(f"Claim reported {delay} days after the failure")
            score += 15

    score = min(score, 100)
    recommendation = "investigate" if score > 40 else "proceed"
    claim["abuse_score"] = score
    claim["red_flags"] = red_flags
    claim["abuse_recommendation"] = recommendation
    claim["status"] = "abuse_screening"

    log_audit(state, "AbuseScreener",
              f"Score: {score}/100. Recommendation: {recommendation}",
              f"Red flags: {red_flags}")
    await ctx.store.set("state", state)
    return f"Abuse screening complete. Score: {score}/100."

**Assessment Agent.** Estimates the repair cost from the diagnosis (a parts catalogue would do this in production), subtracts the service fee the customer pays, caps the result at the plan limit, and flags *replacement* when the repair would cost 70% or more of the invoice value.

In [ ]:
async def assess_repair(ctx: Context) -> str:
    """Estimate the covered repair cost from the diagnosis.

    Used by: AssessmentAgent.
    Reads: fault_description, service_fee, applicable_limit from claim.
    Writes: covered_amount, assessment_breakdown to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]

    # In production: parts catalogue + labour rates; for the lab, keyword rules
    fault = claim.get("fault_description", "").lower()
    fee = claim.get("service_fee", 199)
    limit = claim.get("applicable_limit", 18500)

    estimate = 1500  # default: minor part plus labour
    if "pump" in fault:
        estimate = 2800
    if "compressor" in fault:
        estimate = 9500
    if "solenoid" in fault or "valve" in fault:
        estimate += 650
    if "control board" in fault or "pcb" in fault:
        estimate += 3500
    if "beyond repair" in fault:
        estimate = limit * 0.8

    covered = max(0, min(estimate - fee, limit))
    remedy = "replacement" if estimate >= 0.7 * limit else "repair"
    breakdown = (
        f"Repair estimate: Rs {estimate:,.0f}. "
        f"Service fee paid by customer: Rs {fee:,.0f}. "
        f"Plan limit: Rs {limit:,.0f}. "
        f"Covered by NovaMart: Rs {covered:,.0f}. Remedy: {remedy}."
    )

    claim["covered_amount"] = covered
    claim["assessment_breakdown"] = breakdown
    claim["status"] = "assessment"
    log_audit(state, "AssessmentAgent", f"Covered amount: Rs {covered:,.0f}", breakdown)
    await ctx.store.set("state", state)
    return f"Assessment complete. Covered amount: Rs {covered:,.0f} ({remedy})."

**Decision Agent.** Three-way decision: DENIED if coverage failed on documented grounds, ESCALATED if coverage evidence is incomplete or the abuse score crossed the threshold, otherwise APPROVED. The reasoning is verbose on purpose: it must explain the whole chain.

In [ ]:
async def make_decision(ctx: Context) -> str:
    """Make the claim decision with its reasoning chain.

    Used by: DecisionAgent.
    Reads: coverage, abuse, assessment data from claim.
    Writes: decision, decision_reasoning to claim.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]

    evidence_codes, evidence_complete = coverage_evidence(claim, state.get("plan", {}))
    if not claim.get("coverage_verified", False):
        if evidence_complete and evidence_codes:
            decision = "DENIED"
            reason_codes = evidence_codes
            reasoning = (
                f"Claim denied on recorded coverage evidence: {evidence_codes}. "
                f"{claim.get('coverage_reasoning', '')}"
            )
        else:
            decision = "ESCALATED"
            reason_codes = ["INSUFFICIENT_COVERAGE_EVIDENCE"]
            reasoning = "Coverage could not be established from the supplied evidence; refer for human review."
    elif claim.get("abuse_recommendation") == "investigate":
        decision = "ESCALATED"
        reason_codes = ["ABUSE_REVIEW_REQUIRED"]
        reasoning = (
            f"Claim escalated for abuse investigation. "
            f"Abuse score: {claim.get('abuse_score', 0)}/100. "
            f"Red flags: {claim.get('red_flags', [])}."
        )
    else:
        decision = "APPROVED"
        reason_codes = ["COVERAGE_VERIFIED"]
        reasoning = (
            f"Claim approved. {claim.get('coverage_reasoning', 'Covered by the plan.')} "
            f"Abuse score: {claim.get('abuse_score', 0)}/100 (low risk). "
            f"Covered amount: Rs {claim.get('covered_amount', 0):,.0f}."
        )

    claim["decision"] = decision
    claim["decision_reasoning"] = reasoning
    claim["decision_reason_codes"] = reason_codes
    claim["status"] = decision.lower()
    log_audit(state, "DecisionAgent", f"Decision: {decision}", reasoning)
    await ctx.store.set("state", state)
    return f"Decision: {decision}."

**Compliance Agent (guardrail).** Three checks: a denial must cite reason codes the evidence supports; the reasoning must be long enough to explain to the customer; every pipeline stage must have run. If any check fails, the decision is BLOCKED and goes to a person.

In [ ]:
async def validate_compliance(ctx: Context) -> str:
    """Apply the lab's consumer-protection evidence checks (not a legal certification).

    Used by: ComplianceAgent (guardrail - has veto power).
    Reads: decision, reasoning from claim.
    Writes: status update, compliance issues if found.
    """
    state = await ctx.store.get("state")
    claim = state["claim"]
    issues = []

    # Check 1: a denial must cite reason codes the evidence supports
    if claim.get("decision") == "DENIED":
        evidence_codes, complete = coverage_evidence(claim, state.get("plan", {}))
        codes = claim.get("decision_reason_codes", [])
        if (not complete or not evidence_codes or not isinstance(codes, list)
                or not all(isinstance(code, str) for code in codes)
                or set(codes) != set(evidence_codes)):
            issues.append("Denial requires reason codes supported by the plan and the evidence.")
        if claim.get("coverage_verified") is not False:
            issues.append("Denial contradicts the recorded coverage result.")

    if claim.get("decision") not in {"APPROVED", "DENIED", "ESCALATED"}:
        issues.append("Unrecognized decision value.")
    if claim.get("decision") == "APPROVED":
        codes, complete = coverage_evidence(claim, state.get("plan", {}))
        if not complete or codes or claim.get("coverage_verified") is not True:
            issues.append("Approval requires complete, consistent coverage evidence.")
        if claim.get("abuse_recommendation") == "investigate":
            issues.append("Approval cannot bypass an abuse referral.")

    # Check 2: the customer is owed an explanation they can understand
    if len(claim.get("decision_reasoning", "")) < 50:
        issues.append("Decision reasoning is missing or too short to explain to the customer.")

    # Check 3: every pipeline stage ran
    for field in ["coverage_verified", "abuse_score", "covered_amount"]:
        if claim.get(field) is None:
            issues.append(f"Missing required field: {field}")

    if issues:
        claim["status"] = "compliance_review"
        claim["compliance_issues"] = issues
        log_audit(state, "ComplianceAgent", "BLOCKED - compliance issues", str(issues))
        await ctx.store.set("state", state)
        return f"BLOCKED: {issues}"
    claim["compliance_issues"] = []
    log_audit(state, "ComplianceAgent", "PASSED - lab evidence checks")
    await ctx.store.set("state", state)
    return "Decision validated against the lab's evidence checks."

## 5. Define the seven pipeline agents

Each agent's `system_prompt` is a tight job description and `can_handoff_to` names the **single next agent**. The Compliance agent is the end of the line: it writes the final summary and stops. It *can* hand back to Intake, but only for a malformed claim record; its prompt forbids routine handoffs, which would otherwise loop the pipeline forever.

In [ ]:
from llama_index.core.agent.workflow import FunctionAgent
from llama_index.llms.openai import OpenAI

llm = OpenAI(model='gpt-4o', temperature=0)
ONCE = 'Call your tool exactly once; never call it a second time. '

intake_agent = FunctionAgent(
    name='IntakeAgent', description="Register the customer's warranty claim.",
    system_prompt=ONCE + 'You handle claim intake: call process_intake with the claim data, then hand off to EvidenceParser.',
    llm=llm, tools=[process_intake], can_handoff_to=['EvidenceParser'],
)
parser_agent = FunctionAgent(
    name='EvidenceParser', description='Parse the claim form, photos and technician job sheet.',
    system_prompt=ONCE + 'Parse the submitted evidence with parse_evidence, then hand off to CoverageAgent.',
    llm=llm, tools=[parse_evidence], can_handoff_to=['CoverageAgent'],
)
coverage_agent = FunctionAgent(
    name='CoverageAgent', description='Verify warranty-plan coverage.',
    system_prompt=ONCE + 'Verify coverage with verify_coverage, then hand off to AbuseScreener.',
    llm=llm, tools=[verify_coverage], can_handoff_to=['AbuseScreener'],
)
abuse_agent = FunctionAgent(
    name='AbuseScreener', description='Score the claim for abuse indicators.',
    system_prompt=ONCE + 'Score the claim with screen_for_abuse, then hand off to AssessmentAgent.',
    llm=llm, tools=[screen_for_abuse], can_handoff_to=['AssessmentAgent'],
)
assessment_agent = FunctionAgent(
    name='AssessmentAgent', description='Estimate the covered repair cost.',
    system_prompt=ONCE + 'Estimate the covered amount with assess_repair, then hand off to DecisionAgent.',
    llm=llm, tools=[assess_repair], can_handoff_to=['DecisionAgent'],
)
decision_agent = FunctionAgent(
    name='DecisionAgent', description='Make the claim decision.',
    system_prompt=ONCE + 'Make the decision with make_decision, then hand off to ComplianceAgent.',
    llm=llm, tools=[make_decision], can_handoff_to=['ComplianceAgent'],
)
compliance_agent = FunctionAgent(
    name='ComplianceAgent', description='Consumer-protection guardrail with veto power.',
    system_prompt=(
        ONCE + 'Validate the decision with validate_compliance. You have VETO power. '
        'Whether the result is compliant or BLOCKED, do NOT hand off: write a short final '
        'summary of the claim decision (or the blocking issues) for the customer-care agent and stop. '
        'Hand off to IntakeAgent only if the tool reports that the claim record itself '
        'is fundamentally malformed.'
    ),
    llm=llm, tools=[validate_compliance], can_handoff_to=['IntakeAgent'],
)

## 6. Assemble and run the workflow

We wrap the run in a small function so we can push more than one claim through the same pipeline. The shared state starts with the plan, the evidence, the customer's claim history and the intake date.

In [ ]:
from llama_index.core.agent.workflow import AgentWorkflow


async def run_claim(claim: dict, plan: dict, documents: list, history: list, intake_date: str):
    workflow = AgentWorkflow(
        agents=[intake_agent, parser_agent, coverage_agent, abuse_agent,
                assessment_agent, decision_agent, compliance_agent],
        root_agent='IntakeAgent',
        initial_state={
            'claim': {}, 'plan': plan,
            'submitted_documents': documents,
            'claims_history': history,
            'intake_date': intake_date,
        },
    )
    # Seven agents each need ~3 workflow iterations (tool call, handoff, response),
    # so the default max_iterations=20 is too tight for this pipeline.
    handler = workflow.run(user_msg=json.dumps(claim), max_iterations=60)
    last = None
    async for event in handler.stream_events():
        name = getattr(event, 'current_agent_name', None)
        if name and name != last:
            print(f'[Active: {name}]')
            last = name
    result = await handler
    print('\nFinal summary:\n' + str(result))
    return await handler.ctx.store.get('state')


final_state = await run_claim(SAMPLE_CLAIM, SAMPLE_PLAN, SAMPLE_DOCUMENTS, history=[], intake_date='2026-10-23')

## 7. Inspect the audit trail

We pull the final claim record out of the workflow's context (`handler.ctx.store.get(...)` after the run finishes) and print the audit trail.

In [ ]:
def show_claim(state: dict):
    claim = state.get('claim', {}) if state else {}
    print(f'Claim ID: {claim.get("claim_id")}')
    print(f'Decision: {claim.get("decision")}  {claim.get("decision_reason_codes")}')
    paid = claim.get('decision') == 'APPROVED'
    print(f'Covered:  Rs {claim.get("covered_amount") or 0:,.0f}' + ('' if paid else '  (estimate only: not paid)'))
    print(f'Status:   {claim.get("status")}\n')
    print('=' * 50)
    print('  AUDIT TRAIL')
    print('=' * 50)
    for entry in claim.get('audit_log', []):
        print(f'  {entry["timestamp"]} [{entry["agent"]}] {entry["action"]}')
        if entry.get('detail'):
            print(f'      {entry["detail"][:120]}')


show_claim(final_state)

## 8. A second claim: the denial path

Same plan, different story. The customer reports that the purifier fell during wall
renovation work. The description contains an **excluded cause** (`physical damage`), so the
pipeline should DENY with the reason code `PLAN_EXCLUSION`, and Compliance should *pass* the
denial because the code is backed by the evidence. Watch how the reason codes travel from
CoverageAgent to DecisionAgent to ComplianceAgent.

In [ ]:
second_claim = {
    **SAMPLE_CLAIM,
    'date_of_failure': '2026-10-15',
    'description': ('Purifier fell off its wall bracket during renovation work in the kitchen. '
                    'Casing cracked and leaking: physical damage to the unit.'),
}
second_state = await run_claim(second_claim, SAMPLE_PLAN, SAMPLE_DOCUMENTS, history=[], intake_date='2026-10-16')
show_claim(second_state)

## 9. Results

You processed a warranty claim through seven specialist agents and produced a fully
auditable decision. **What to notice about the claims-pipeline architecture:**

* **Compliance is an agent, not an afterthought.** A guardrail agent with veto power gives you
  one testable place to enforce consumer-protection rules.
* **Audit trails are written by tools, not by agents.** Every tool calls `log_audit` as it
  changes state, so the trail is a by-product, not extra work.
* **The LLM routes; Python decides.** Coverage, abuse score, amount and decision are
  deterministic code. The LLM only moves the claim along and writes the summary. That is what
  makes the outcome explainable to a customer and to a regulator.
* **The same architecture fits returns, installations or refunds**: swap the tools, keep the
  topology.

**Discussion questions for class**

1. Which of the seven steps would you keep fully automated, and which need a human for claims above Rs 10,000?
2. The abuse screener uses three hand-picked red flags. Who should own those rules, and how would you check they do not unfairly target some customer groups?
3. What would you show Ananya, in plain language, from this audit trail?